In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf

In [2]:
# tickers = ["AAPL", "MSFT", "GOOGL", "AMZN", "AMD"]
tickers = ['TLT', 'GLD', 'SPY', 'QQQ', 'VWO']
start = "2019-01-01"
risk_free_rate = 0.02

n_samples = 5*1000
n_itr = 5*1000
mutation_prob = 0.1

data = yf.download(tickers, start=start)["Close"]
returns = data.pct_change().dropna()

mean = returns.mean() * 252
cvar = returns.cov() * 252

n_assets = len(tickers)

[*********************100%***********************]  5 of 5 completed


In [3]:
def portfolio_return(w):
	return np.dot(w, mean)

def portfolio_volatility(w):
	return np.sqrt(w.T @ cvar @ w)

def sharpe_ratio(w):
	r = portfolio_return(w)
	v = portfolio_volatility(w)
	return (r - risk_free_rate) / v

In [4]:
def gen_sample(sample_count, asset_count):
	samples = np.random.dirichlet(np.ones(asset_count), size=sample_count)
	for i in range(len(samples)):
		samples[i] /= np.sum(samples[i])
	return samples

def max_sharpe_and_weights(sample_space):
	sample_sharpe = np.array([sharpe_ratio(w) for w in sample_space])
	max_sharpe = max(sample_sharpe)
	best_weight = sample_space[np.argmax(max_sharpe)].copy()
	return max_sharpe, best_weight

In [5]:
def random_walks(n_sample_spaces, itr, n_samples, n_asset, n_particle):
	noise = 0.1
	gBest = []
	mutation_prob = 0.2
	for s in range(n_sample_spaces):
		samples = gen_sample(sample_count=n_samples, asset_count=n_asset)
		indices = np.random.choice(len(samples), size=n_particle, replace=False)
		particle = samples[indices].copy()
		# particle = np.random.dirichlet(np.ones(n_asset), size=n_particle)
		velocity = np.random.randn(n_particle, n_asset) * 0.2
		pBest = particle.copy()
		pBest_weight = np.array([sharpe_ratio(w) for w in particle])
		itr_best = pBest[np.argmax(pBest_weight)].copy()
		for _ in range(itr):
			weights = np.random.uniform(0.4, 0.9)
			c1 = np.random.uniform(1.0, 2.5)
			c2 = np.random.uniform(1.0, 2.5)

			for i in range(n_particle):
				r1 = np.random.rand(n_asset)
				r2 = np.random.rand(n_asset)
				velocity[i] = (
					weights * velocity[i]
					+ c1 * r1 * (pBest[i] - particle[i])
					+ c2 * r2 * (itr_best - particle[i])
				)
				velocity[i] += np.random.normal(0, noise, n_asset)
				particle[i] += velocity[i]
				if np.random.rand() < mutation_prob:
					idx = np.random.randint(len(samples))
					particle[i] = samples[idx].copy()
				particle[i] = np.maximum(particle[i], 0)
				total = np.sum(particle[i])
				if total <= 1e-12 :
					idx = np.random.randint(len(samples))
					particle[i] = samples[idx].copy()
				particle[i] /= np.sum(particle[i])

				score = sharpe_ratio(particle[i])
				if score > pBest_weight[i]:
					pBest[i] = particle[i].copy()
					pBest_weight[i] = score
			k = max(5, n_particle // 3)
			top_indices = np.argsort(pBest_weight)[-k:]
			chosen = np.random.choice(top_indices)
			itr_best = pBest[chosen].copy()
		gBest.append(itr_best)
	return gBest

In [6]:
gbest = random_walks(20, n_itr, n_samples, n_assets, 40)
gbest = np.array(gbest)

best_return_sharpe = 0
best_return_weights = []
run : int = 0
for i in gbest:
	run += 1
	print("--------------------------")
	print(f"-----Simulation {run} Run-----")
	print("Randomized PSO Portfolio")
	print("--------------------------")
	for t, w in zip(tickers, i):
		print(f"{t}: {w}")

	ret = portfolio_return(i)
	vol = portfolio_volatility(i)
	sharpe = sharpe_ratio(i)
	if sharpe > best_return_sharpe:
		best_return_sharpe = sharpe
		best_return_weights = i

	print("\nMetrics")
	print("--------------------------")
	print(f"Expected Return: {ret*100}%")
	print(f"Volatility: {vol*100}%")
	print(f"Sharpe Ratio: {sharpe}")

print("--------------------------")
print("---Best Simulation Run----")
print("Randomized PSO Portfolio")
print("--------------------------")
for t, w in zip(tickers, best_return_weights):
	print(f"{t}: {w}")

ret = portfolio_return(best_return_weights)
vol = portfolio_volatility(best_return_weights)
sharpe = sharpe_ratio(best_return_weights)

print("\nMetrics")
print("--------------------------")
print(f"Expected Return: {ret*100}%")
print(f"Volatility: {vol*100}%")
print(f"Sharpe Ratio: {sharpe}")

--------------------------
-----Simulation 1 Run-----
Randomized PSO Portfolio
--------------------------
TLT: 0.6348690429855138
GLD: 0.36513095701448617
SPY: 0.0
QQQ: 0.0
VWO: 0.0

Metrics
--------------------------
Expected Return: 19.57327730628389%
Volatility: 14.78678306112424%
Sharpe Ratio: 1.1884449263670875
--------------------------
-----Simulation 2 Run-----
Randomized PSO Portfolio
--------------------------
TLT: 0.6348565499837658
GLD: 0.36514345001623416
SPY: 0.0
QQQ: 0.0
VWO: 0.0

Metrics
--------------------------
Expected Return: 19.57331297664045%
Volatility: 14.78681307382944%
Sharpe Ratio: 1.1884449264962116
--------------------------
-----Simulation 3 Run-----
Randomized PSO Portfolio
--------------------------
TLT: 0.6348508349972736
GLD: 0.3651491650027264
SPY: 0.0
QQQ: 0.0
VWO: 0.0

Metrics
--------------------------
Expected Return: 19.57332929422449%
Volatility: 14.786826806010456%
Sharpe Ratio: 1.1884449263368253
--------------------------
-----Simulation 4 R